# COMP663 Assignment 2 — Classical Optimisation

**Student ID:** 1173808  
**Dataset:** `forest_cover_data.csv`  
**Primary metric:** macro-F1


## ENV & Libs Setup


In [27]:
from pathlib import Path
import ast
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import (
    ParameterSampler,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
DEVICE = torch.device("cpu")
torch.set_num_threads(4)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "forest_cover_data.csv"
FIGURE_PATH = ROOT / "figures" / "performance_comparison.png"
MODEL_PATH = ROOT / "models" / "1173808_Assignment2_final.pt"

TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
SEARCH_EPOCHS = 20
FINAL_EPOCHS = 100
RANDOM_TRIALS = 6
BAYESIAN_TRIALS = 8
NAS_TRIALS = 8

print(f"env ready: torch={torch.__version__}, device={DEVICE}, seed={SEED}")

env ready: torch=2.13.0, device=cpu, seed=42


## Task 1 — Baseline model and candidate hyperparameters

### 1.1 Preprocessing pipeline and feature engineering

As we found during assginment 1:

| Decision                 | Evidence from EDA                                                                                                                      | Action                                                                                                       | Reason / trade-off                                                                                                                                                                                              |
| ------------------------ | -------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Missing values           | Zero missing values across all 15 columns (confirmed in Task 1.1)                                                                      | pass through                                                                                                 | Imputing when there is nothing to impute would add unnecessary complexity and risk introducing artificial patterns.                                                                                             |
| Scaling / transformation | Distance features have large ranges and outliers (Task 1.3)                                                                            | Apply `StandardScaler` to the 10 continuous features and leave the 4 binary Wilderness_Area columns unscaled | Scaling puts continuous features on a comparable range. The fitted transformation stays inside the validation pipeline, which prevents leakage. Some models are notdistance-sensitive which will discuss later. |
| Feature engineering      | Elevation, distance features, and wilderness area already show useful class separation; wilderness columns are already one-hot encoded | No additional feature engineering                                                                            | The existing features already contain useful information. Adding polynomial features would increase complexity without EDA evidence that they are needed.                                                       |
| Class imbalance          | The largest and smallest classes have a 31:1 ratio (Task 1.2)                                                                          | Use class weighting and evaluate with macro-F1                                                               | Class weighting gives more importance to rare classes without changing the training data. Macro-F1 makes minority-class performance visible.                                                                    |




#### Load data and check data integrity


In [28]:
# load data from CSV file
data = pd.read_csv(DATA_PATH)
target = "Cover_Type"

# filter out rows with missing values in the target column
data = data.dropna(subset=[target])
feature_names = [column for column in data.columns if column != target]
continuous_features = [
    column for column in feature_names if not column.startswith("Wilderness_Area")
]

# check data integrity
assert data.shape == (571_012, 15), data.shape
assert len(feature_names) == 14
assert set(data[target].unique()) == {1, 2, 3, 4, 5}
assert data.isna().sum().sum() == 0

# print data shape, feature names, and target value counts with percentages
print("Shape:", data.shape)
print("Features:", feature_names)
display(
    data[target]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
    .assign(percentage=lambda frame: 100 * frame["count"] / len(data))
)

Shape: (571012, 15)
Features: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area1', 'Wilderness_Area2', 'Wilderness_Area3', 'Wilderness_Area4']


,count,percentage
Cover_Type,,
1,209840,36.748790
2,317055,55.525103
3,10240,1.793307
4,15367,2.691187
5,18510,3.241613


#### Data set split

In [29]:
# split data into training, validation, and test sets
train_validation_frame, test_frame = train_test_split(
    data, test_size=TEST_FRACTION, stratify=data[target], random_state=SEED
)

train_frame, validation_frame = train_test_split(
    train_validation_frame,
    test_size=VALIDATION_FRACTION / (TRAIN_FRACTION + VALIDATION_FRACTION),
    stratify=train_validation_frame[target],
    random_state=SEED,
)

# validate the size of the splits and print the number of samples in each set
assert len(train_frame) + len(validation_frame) + len(test_frame) == len(data)
print(
    f"Train / validation / test: {len(train_frame):,} / {len(validation_frame):,} / {len(test_frame):,}"
)

Train / validation / test: 342,606 / 114,203 / 114,203


### 1.2 Primary and secondary evaluation metrics

As this dataset have a strong imblanaced class,

**Macro-F1** will be the primary metric to cover type has equal importance.

**Balanced accuracy** is the secondary metric because it reflect the model recalls on each class equally.


### 1.3 Baseline model training and evaluation procedure

- (1) Initial the baseline mode with configuration in assignment requirement.

- (2) traning baseline model and compute metrics on validation dataset.


In [ ]:
# define the baseline neural network architecture
class BaselineNN(nn.Module):
    """The architecture supplied in baselineNN.ipynb."""

    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 24),
            nn.Sigmoid(),
            nn.Linear(24, 12),
            nn.Sigmoid(),
            nn.Linear(12, 5),
        )

    def forward(self, x):
        return self.layers(x)


baseline_model = BaselineNN(len(feature_names)).to(DEVICE)
print(baseline_model)
print(
    f"Parameters: {sum(parameter.numel() for parameter in baseline_model.parameters()):,}"
)
# check that the number of parameters in the baseline model matches the expected count 14*24+24 + 24*12+12 + 12*5+5 = 725
assert (
    sum(parameter.numel() for parameter in baseline_model.parameters()) == 725
), "Parameter count mismatch"


BaselineNN(
  (layers): Sequential(
    (0): Linear(in_features=14, out_features=24, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=24, out_features=12, bias=True)
    (3): Sigmoid()
    (4): Linear(in_features=12, out_features=5, bias=True)
  )
)
Parameters: 725


In [31]:
# train the baseline model and evaluate metrics
BASELINE_CONFIG = {
    "learning_rate": 1e-3,
    "batch_size": 512,
    "weight_decay": 1e-4,
    "epochs": SEARCH_EPOCHS,
}

scaler = StandardScaler().fit(train_frame[continuous_features])


def prepare_baseline_data(frame):
    features = frame[feature_names].astype("float32").copy()
    features[continuous_features] = scaler.transform(features[continuous_features])
    return features.to_numpy(), frame[target].to_numpy(dtype=np.int64) - 1


x_train, y_train = prepare_baseline_data(train_frame)
x_validation, y_validation = prepare_baseline_data(validation_frame)

class_counts = np.bincount(y_train, minlength=5)
class_weights = len(y_train) / (5 * class_counts)
loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)
optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=BASELINE_CONFIG["learning_rate"],
    weight_decay=BASELINE_CONFIG["weight_decay"],
)

x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

baseline_model.train()
for _ in range(BASELINE_CONFIG["epochs"]):
    order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
    for start in range(0, len(order), BASELINE_CONFIG["batch_size"]):
        batch_index = order[start : start + BASELINE_CONFIG["batch_size"]]
        optimizer.zero_grad()
        loss = loss_fn(
            baseline_model(x_train_tensor[batch_index]), y_train_tensor[batch_index]
        )
        loss.backward()
        optimizer.step()

baseline_model.eval()
with torch.no_grad():
    validation_logits = baseline_model(
        torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)
    )
validation_prediction = validation_logits.argmax(dim=1).cpu().numpy()

validation_macro_f1 = f1_score(y_validation, validation_prediction, average="macro")
validation_balanced_accuracy = balanced_accuracy_score(
    y_validation, validation_prediction
)
assert 0 <= validation_macro_f1 <= 1
print(f"Validation macro-F1: {validation_macro_f1:.4f}")
print(f"Validation balanced accuracy: {validation_balanced_accuracy:.4f}")

Validation macro-F1: 0.4628
Validation balanced accuracy: 0.7557


### 1.4 Unoptimised baseline performance using held-out test data


### 1.5 Candidate hyperparameters


### 1.6 Hyperparameter types and search ranges


## Task 2 — Classical hyperparameter search


### 2.1 Selected classical search method and justification


### 2.2 Hyperparameters to optimise


### 2.3 Computational budget and justification


### 2.4 Apply the selected search method


### 2.5 Evaluation procedure and primary metric


### 2.6 Best configuration, score, and search time


### 2.7 Search results table


## Task 3 — Bayesian optimisation


### 3.1 Hyperparameters to optimise


### 3.2 Objective, search space, surrogate model, and acquisition process


### 3.3 Evaluation procedure and primary metric


### 3.4 Computational budget and justification


### 3.5 Apply Bayesian optimisation


### 3.6 Best configuration, performance, and search time


### 3.7 Search results table


## Task 4 — Neural architecture search


### 4.1 NAS hyperparameters to optimise


### 4.2 Objective, NAS search space, and search strategy


### 4.3 Architecture design choices


### 4.4 Evaluation procedure and primary metric


### 4.5 Computational budget and justification


### 4.6 Apply NAS to hyperparameters and architecture


### 4.7 Best configuration, performance, and search time


### 4.8 Search results table


## Task 5 — Performance comparison and analysis


### 5.1 Compare primary and secondary metrics


### 5.2 Search time, trials, and model complexity


### 5.3 Summary table and comparison visualisation


### 5.4 Hyperparameter or architecture effects


### 5.5 Fairness of the comparison


### 5.6 Final model selection and complete configuration


### 5.7 Class-level evaluation, limitations, and saved model


## Task 6 — Hidden test


### 6.1 Run the final model on a hidden-test CSV and create Cover_Type predictions
